# FruitBlend24 วิเคราะห์ยอดขายและกำไร
ข้อมูล ก.ย. 2025–ส.ค. 2026 และประมาณการ ก.ย.–พ.ย. 2026 หน่วยเงินเป็นบาท

ครั้งแรกให้รันจากบนลงล่าง: Excel → คลีนและบันทึก cleaned_data → อ่าน CSV เพื่อวิเคราะห์ → app_data → Streamlit
เมื่อมี cleaned_data ครบแล้ว สามารถเริ่มรันจากหัวข้อ 2 ได้ แม้ Restart Kernel แล้ว

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.float_format', '{:,.2f}'.format)
file = Path('FruitBlend24_Intern_Case_Data.xlsx')
sheets = pd.read_excel(file, sheet_name=None)
raw = sheets['orders_hourly'].copy()
print(f'ยอดขายดิบ {len(raw):,} แถว จาก {len(sheets)} ชีต')

ยอดขายดิบ 124,397 แถว จาก 11 ชีต


---

## 1. Data quality
* ตัดแถวซ้ำ รวมชื่อเป็น 5 เมนู และแยก 3 รายการนอกขอบเขต  
* ต้นทุน Mixed Berry วันเปิดขายขาด ใช้ต้นทุนสัปดาห์แรกที่มี: ผลไม้ 33.69 และแพ็กเกจ 8 บาท/แก้ว พร้อมติดธง  
* เติมของเสีย 6 รายการด้วยค่ามัธยฐานของครัว/เมนู/วันในสัปดาห์เดียวกัน ช่วงก่อนและหลัง 4 สัปดาห์แล้วคูณต้นทุนผลไม้และแพ็กเกจของสัปดาห์นั้น   ปัดต้นทุนแต่ละรายการเป็นสตางค์แบบครึ่งขึ้น พร้อมระบุ waste_estimated  
* ใช้ข้อมูลหลังวันที่ขาดเพื่อเติมประวัติย้อนหลังเท่านั้น ไม่ใช้ในขั้นทดสอบพยากรณ์
README และ data_dictionary ใน Excel ระบุว่ามี exact duplicates และยอดเป็นระดับชั่วโมง จึงตัดแถวที่เหมือนกันครบทุกคอลัมน์เป็นกรณีหลัก พร้อมแสดงผลกรณีเก็บแถวเหล่านี้ในข้อ 4 โดยคงของเสียและ overhead เท่าเดิม

In [2]:
groups = {
    'Watermelon': ['watermelon ice','watermelon smoothie','watermelon smoothy','แตงโมปั่น','แตงโมปั่นสด','แตงโมปั่น (ไซส์ m)'],
    'Pineapple': ['pineapple ice','pineapple smoothie','สับปะรดปั่น','สัปปะรดปั่น'],
    'Guava': ['guava ice','guava smoothie','ฝรั่งปั่น','ฝรั่งปั่น (ใส่เกลือ)'],
    'PassionFruit': ['passion fruit ice','passion fruit smoothie','passionfruit ice','เสาวรสปั่น','เสาวรสปั่นเข้มข้น'],
    'MixedBerryPremium': ['mixed berry (premium)','mixed berry premium','berry premium smoothie','เบอร์รี่ปั่นพรีเมียม','เบอร์รี่รวมพรีเมียม']
}
mapping = {name: sku for sku, names in groups.items() for name in names}
duplicates = raw.loc[raw.duplicated()].copy()
orders = raw.drop_duplicates().copy()
orders['name_key'] = orders['item_name_raw'].str.strip().str.casefold()
orders['sku'] = orders['name_key'].map(mapping)
excluded = orders.loc[orders['name_key'].isin(['combo set a (2 แก้ว)','น้ำส้มปั่น (เมนูทดลอง)','แก้วเปล่า'])].copy()
orders = orders.drop(index=excluded.index).copy()
assert orders['sku'].notna().all(), 'มีชื่อเมนูที่ยังไม่ได้จับคู่'
orders['date'] = pd.to_datetime(orders['date'])
orders['month'] = orders['date'].dt.to_period('M').dt.to_timestamp()
orders['week_start'] = orders['date'] - pd.to_timedelta(orders['date'].dt.dayofweek, unit='D')
parts = orders['rate_code'].str.split('-', expand=True)
orders['platform_code'], orders['base_code'] = parts[0], parts[1]
bridge = pd.DataFrame({
    'รายการ':['ข้อมูลดิบ','แถวซ้ำที่ตัด','รายการนอกขอบเขต','ข้อมูลวิเคราะห์'],
    'แถว':[len(raw),len(duplicates),len(excluded),len(orders)],
    'รายได้':[raw['gross_revenue_thb'].sum(),duplicates['gross_revenue_thb'].sum(),excluded['gross_revenue_thb'].sum(),orders['gross_revenue_thb'].sum()]
})
display(bridge)

,รายการ,แถว,รายได้
0,ข้อมูลดิบ,124397,"31,276,898.60"
1,แถวซ้ำที่ตัด,372,"91,488.03"
2,รายการนอกขอบเขต,370,"38,478.00"
3,ข้อมูลวิเคราะห์,123655,"31,146,932.57"


In [3]:
# เชื่อมต้นทุนตามเมนู สัปดาห์ และแพลตฟอร์ม
costs = sheets['cost_assumptions']
unit_costs = costs.loc[costs['sku'].isin(groups)].copy()
columns = ['packaging_cost_per_cup_thb','labor_cost_per_cup_thb','base_price_thb']
unit_costs[columns] = unit_costs[columns].apply(pd.to_numeric)
overhead = costs.loc[costs['sku'].isin(orders['kitchen'].unique()), ['sku','packaging_cost_per_cup_thb']].rename(columns={'sku':'kitchen','packaging_cost_per_cup_thb':'overhead_thb'})
overhead['overhead_thb'] = pd.to_numeric(overhead['overhead_thb'])
fruit = sheets['weekly_fruit_cost'].copy()
fruit['week_start'] = pd.to_datetime(fruit['week_start'])
orders = orders.merge(fruit,on=['sku','week_start'],how='left',validate='many_to_one')
orders = orders.merge(unit_costs,on='sku',how='left',validate='many_to_one')
orders = orders.merge(sheets['platform_commission_rate'],on='platform_code',how='left',validate='many_to_one')
orders = orders.merge(sheets['rate_code_dim'],on='base_code',how='left',validate='many_to_one')
orders['estimated_cost'] = orders['fruit_cost_per_cup_thb'].isna() & orders['sku'].eq('MixedBerryPremium') & orders['date'].eq('2026-03-01')
launch_cost = fruit.loc[fruit['sku'].eq('MixedBerryPremium') & fruit['week_start'].eq('2026-03-02'),'fruit_cost_per_cup_thb'].item()
orders.loc[orders['estimated_cost'],'fruit_cost_per_cup_thb'] = launch_cost
assert orders[columns+['fruit_cost_per_cup_thb','commission_rate']].notna().all().all(), 'ต้นทุนจับคู่ไม่ครบ'
for total, rate in [('fruit_cost','fruit_cost_per_cup_thb'),('pack_cost','packaging_cost_per_cup_thb'),('labor_cost','labor_cost_per_cup_thb')]:
    orders[total] = orders['units_sold'] * orders[rate]
orders['commission'] = orders['gross_revenue_thb'] * orders['commission_rate']
orders['profit_before_waste'] = orders['gross_revenue_thb'] - orders[['fruit_cost','pack_cost','labor_cost','commission']].sum(axis=1)

# จัดการต้นทุนของเสียที่ขาด และเติมวันที่ไม่มีบันทึกด้วยค่าประมาณ
waste = sheets['waste_daily'].copy()
waste['date'] = pd.to_datetime(waste['date'])
waste['month'] = waste['date'].dt.to_period('M').dt.to_timestamp()
waste['estimated_cost'] = waste['est_waste_cost_thb'].isna() & waste['sku'].eq('MixedBerryPremium') & waste['date'].eq('2026-03-01')
waste.loc[waste['estimated_cost'],'est_waste_cost_thb'] = waste.loc[waste['estimated_cost'],'units_wasted'] * (launch_cost+8)
assert waste['est_waste_cost_thb'].notna().all(), 'ยังมีต้นทุนของเสียที่ขาด'
keys = ['date','kitchen','sku']
coverage = orders[keys].drop_duplicates().merge(waste[keys],on=keys,how='left',indicator=True,validate='one_to_one')
missing_waste = coverage.loc[coverage['_merge'].eq('left_only'),keys].copy()
# เติมของเสียจากวันเดียวกันของสัปดาห์ในช่วงก่อน/หลัง 4 สัปดาห์
waste_estimates = missing_waste.copy()
for i, row in missing_waste.iterrows():
    reference = waste.loc[
        waste['kitchen'].eq(row['kitchen']) & waste['sku'].eq(row['sku'])
        & waste['date'].dt.dayofweek.eq(row['date'].dayofweek)
        & waste['date'].between(row['date']-pd.Timedelta(days=28), row['date']+pd.Timedelta(days=28))
    ]
    assert not reference.empty, 'ไม่มีข้อมูลอ้างอิงสำหรับประมาณของเสีย'
    waste_estimates.loc[i, 'units_wasted'] = reference['units_wasted'].median()
    waste_estimates.loc[i, 'reference_days'] = len(reference)
waste_estimates['week_start'] = waste_estimates['date'] - pd.to_timedelta(waste_estimates['date'].dt.dayofweek, unit='D')
waste_estimates = waste_estimates.merge(fruit, on=['sku','week_start'], validate='many_to_one')
waste_estimates = waste_estimates.merge(unit_costs[['sku','packaging_cost_per_cup_thb']], on='sku', validate='many_to_one')
waste_estimates['est_waste_cost_thb'] = np.floor(waste_estimates['units_wasted'] * (waste_estimates['fruit_cost_per_cup_thb']+waste_estimates['packaging_cost_per_cup_thb']) * 100 + .5) / 100
waste_estimates['month'] = waste_estimates['date'].dt.to_period('M').dt.to_timestamp()
waste_estimates['estimated_cost'], waste_estimates['waste_estimated'] = True, True
waste['waste_estimated'] = waste['estimated_cost']
waste = pd.concat([waste, waste_estimates], ignore_index=True)
waste_adjustment = waste_estimates['est_waste_cost_thb'].sum()
display(waste_estimates[['date','kitchen','sku','units_wasted','reference_days','est_waste_cost_thb','waste_estimated']])
print(f'เติมต้นทุนของเสีย {len(waste_estimates)} รายการ รวม {waste_adjustment:,.2f} บาท เข้า P&L')

quality = pd.DataFrame({
    'ปัญหา':['แถวซ้ำ','รายการนอกเมนู','ชื่อสินค้าหลายแบบ','ต้นทุนผลไม้ที่ประมาณ','ต้นทุนของเสียที่ประมาณ','บันทึกของเสียที่เติมด้วยค่าประมาณ'],
    'จำนวน':[len(duplicates),len(excluded),raw['item_name_raw'].nunique(),orders['estimated_cost'].sum(),waste['estimated_cost'].sum(),len(missing_waste)]
})
display(quality)
# เก็บข้อมูลพร้อมวิเคราะห์ แยกจากผลสรุปที่แอปใช้
cleaned = Path('cleaned_data')
review = cleaned / 'review'
review.mkdir(parents=True, exist_ok=True)
clean_tables = {
    'orders_clean': orders, 'waste_clean': waste, 'weekly_fruit_cost': fruit,
    'menu_costs': unit_costs, 'kitchen_overhead': overhead,
    'platform_commission_rate': sheets['platform_commission_rate'],
    'rate_code_dim': sheets['rate_code_dim'], 'monthly_budget': sheets['monthly_budget'],
    'platform_dim': sheets['platform_dim'], 'sku_code_dim': sheets['sku_code_dim']
}
for name, table in clean_tables.items():
    table.to_csv(cleaned/f'{name}.csv', index=False, encoding='utf-8-sig')
review_tables = {
    'duplicates': duplicates.rename_axis('source_index').reset_index(),
    'excluded_items': excluded.rename_axis('source_index').reset_index(),
    'waste_estimates': waste_estimates, 'data_quality': quality
}
for name, table in review_tables.items():
    table.to_csv(review/f'{name}.csv', index=False, encoding='utf-8-sig')
print('บันทึกข้อมูลพร้อมวิเคราะห์ไว้ที่', cleaned.resolve())


เติมต้นทุนของเสีย 6 รายการ รวม 418.67 บาท เข้า P&L
บันทึกข้อมูลพร้อมวิเคราะห์ไว้ที่ C:\Users\thewh\OneDrive - KMITL\แบบทดสอบ DABA (Intern)  Atmind Group\cleaned_data


,date,kitchen,sku,units_wasted,reference_days,est_waste_cost_thb,waste_estimated
0,2025-11-26,BKK_Sukhumvit,Guava,3.50,8.00,50.61,True
1,2026-03-16,Pattaya_Beach,Guava,5.00,8.00,77.35,True
2,2025-09-12,BKK_Sukhumvit,Guava,3.00,5.00,46.41,True
3,2025-10-13,Pattaya_Beach,PassionFruit,5.00,8.00,116.10,True
4,2026-07-23,BKK_Ladprao,Watermelon,4.50,8.00,61.25,True
5,2026-05-19,BKK_Ladprao,Watermelon,5.00,8.00,66.95,True


,ปัญหา,จำนวน
0,แถวซ้ำ,372
1,รายการนอกเมนู,370
2,ชื่อสินค้าหลายแบบ,31
3,ต้นทุนผลไม้ที่ประมาณ,75
4,ต้นทุนของเสียที่ประมาณ,10
5,บันทึกของเสียที่เติมด้วยค่าประมาณ,6


--- 

## 2. Demand & pricing
ดูยอดเฉลี่ยต่อวันเพื่อเปรียบเทียบเดือนที่จำนวนวันต่างกัน และใช้ต้นทุน ส.ค. ประเมินราคาปัจจุบัน
ราคาคุ้มทุนรวมต้นทุนผันแปรและของเสีย แต่ยังไม่รวม overhead โดยสมมติภาระของเสียต่อแก้วเท่ากันระหว่างแพลตฟอร์ม
ตั้งแต่หัวข้อนี้อ่านข้อมูลจาก cleaned_data เท่านั้น หากมี CSV ครบแล้ว เริ่มรันเซลล์ด้านล่างได้โดยไม่ต้องอ่าน Excel หรือคลีนซ้ำ
กราฟรายชั่วโมงแยก 4 ครัว เพื่อไม่ใช้รูปแบบรวมไปวางแผนรายครัว จำนวนพนักงานยังต้องมีข้อมูลเวลาทำต่อแก้วและกำลังผลิต

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.float_format', '{:,.2f}'.format)
cleaned = Path('cleaned_data')
orders = pd.read_csv(cleaned/'orders_clean.csv', parse_dates=['date','month','week_start'], encoding='utf-8-sig')
waste = pd.read_csv(cleaned/'waste_clean.csv', parse_dates=['date','month','week_start'], encoding='utf-8-sig')
fruit = pd.read_csv(cleaned/'weekly_fruit_cost.csv', parse_dates=['week_start'], encoding='utf-8-sig')
unit_costs = pd.read_csv(cleaned/'menu_costs.csv', encoding='utf-8-sig')
overhead = pd.read_csv(cleaned/'kitchen_overhead.csv', encoding='utf-8-sig')
platforms = pd.read_csv(cleaned/'platform_commission_rate.csv', encoding='utf-8-sig')
rate_codes = pd.read_csv(cleaned/'rate_code_dim.csv', encoding='utf-8-sig')
budget = pd.read_csv(cleaned/'monthly_budget.csv', parse_dates=['month'], encoding='utf-8-sig')
platform_dim = pd.read_csv(cleaned/'platform_dim.csv', encoding='utf-8-sig')
sku_code_dim = pd.read_csv(cleaned/'sku_code_dim.csv', encoding='utf-8-sig')
quality = pd.read_csv(cleaned/'review/data_quality.csv', encoding='utf-8-sig')
waste_estimates = pd.read_csv(cleaned/'review/waste_estimates.csv', parse_dates=['date','month','week_start'], encoding='utf-8-sig')
missing_waste = waste_estimates[['date','kitchen','sku']].copy()
waste_adjustment = waste_estimates['est_waste_cost_thb'].sum()
print(f'อ่านยอดขาย {len(orders):,} แถว และของเสีย {len(waste):,} แถว จาก cleaned_data')

# เตรียมโฟลเดอร์ผลลัพธ์และข้อมูลคุณภาพที่อ่านจาก cleaned_data
folder = Path('app_data')
folder.mkdir(exist_ok=True)
quality_notes = []
quality_notes.extend({'section':'quality','item':row['ปัญหา'],'value':row['จำนวน']} for _,row in quality.iterrows())
quality_notes.extend({'section':'estimated_waste','item':row['date'].strftime('%Y-%m-%d'),'value':row['kitchen']+' / '+row['sku'],
              'units_wasted':row['units_wasted'],'est_waste_cost_thb':row['est_waste_cost_thb'],
              'reference_days':row['reference_days'],'waste_estimated':True} for _,row in waste_estimates.iterrows())
quality_notes.append({'section':'waste_adjustment','item':'cost_thb','value':waste_adjustment})


อ่านยอดขาย 123,655 แถว และของเสีย 6,576 แถว จาก cleaned_data


In [5]:
daily = orders.groupby(['date','kitchen','sku'],as_index=False)['units_sold'].sum()
daily['month'] = daily['date'].dt.to_period('M').dt.to_timestamp()
monthly_demand = daily.groupby(['month','sku'],as_index=False).agg(units=('units_sold','sum'),days=('date','nunique'))
monthly_demand['cups_per_day'] = monthly_demand['units']/monthly_demand['days']
px.line(monthly_demand,x='month',y='cups_per_day',color='sku',markers=True,title='ยอดขายเฉลี่ยต่อวัน รวมทุกครัว').show()
recent = orders.loc[orders['date'].ge(orders['date'].max()-pd.Timedelta(days=55))]
hourly = (recent.groupby(['kitchen','hour'])['units_sold'].sum()/56).reset_index(name='cups_per_day')
px.bar(hourly,x='hour',y='cups_per_day',facet_col='kitchen',facet_col_wrap=2,
       title='ยอดขายเฉลี่ยแต่ละชั่วโมง แยกครัว: 8 สัปดาห์ล่าสุด').show()

latest_month = orders['month'].max()
latest = orders.loc[orders['month'].eq(latest_month)]
prices = latest.groupby('sku',as_index=False).agg(units=('units_sold','sum'),revenue=('gross_revenue_thb','sum'),fruit=('fruit_cost','sum'),pack=('pack_cost','sum'),labor=('labor_cost','sum'),commission=('commission','sum'))
latest_waste = waste.loc[waste['month'].eq(latest_month)]
prices = prices.merge(latest_waste.groupby('sku',as_index=False).agg(waste_cost=('est_waste_cost_thb','sum')),on='sku',validate='one_to_one')
prices = prices.merge(unit_costs[['sku','base_price_thb']],on='sku',validate='one_to_one')
prices['avg_price'] = prices['revenue']/prices['units']
prices['variable_cost'] = prices[['fruit','pack','labor']].sum(axis=1)/prices['units']
prices['waste_per_cup'] = prices['waste_cost']/prices['units']
prices['profit_per_cup'] = (prices['revenue']-prices['commission']-prices['waste_cost'])/prices['units']-prices['variable_cost']
fees = platforms.set_index('platform_code')['commission_rate']
prices['break_even_LM'] = (prices['variable_cost']+prices['waste_per_cup'])/(1-fees['LM'])
prices['break_even_Grab'] = (prices['variable_cost']+prices['waste_per_cup'])/(1-fees['GB'])
display(prices[['sku','base_price_thb','avg_price','profit_per_cup','break_even_LM','break_even_Grab']])
# บันทึกยอดขายรายชั่วโมง
hourly.to_csv(folder/'hourly.csv', index=False, encoding='utf-8-sig')


Plotly figure

Plotly figure

,sku,base_price_thb,avg_price,profit_per_cup,break_even_LM,break_even_Grab
0,Guava,49.00,47.72,12.27,31.90,29.38
1,MixedBerryPremium,89.00,86.58,-17.07,114.53,105.49
2,PassionFruit,65.00,63.33,2.88,61.63,56.76
3,Pineapple,49.00,47.79,13.04,31.17,28.71
4,Watermelon,45.00,43.85,12.60,27.72,25.53


---

## 3. Promotions & rate codes
สรุปผลจริงแยกเมนู แพลตฟอร์ม และโปร แล้วประเมินกำไรต่อแก้วด้วยต้นทุน ส.ค. บนฐานเดียวกัน
ใช้ส่วนต่างต่อแก้วช่วยเลือกโปรที่ควรปรับ ไม่สรุปว่าโปรทำให้ยอดขายเพิ่มจากยอดรวม เพราะวันและกลุ่มลูกค้าแตกต่างกัน

In [6]:
promo_history = orders.groupby(['sku','platform_name','rate_code_name'],as_index=False).agg(
    active_days=('date','nunique'),units=('units_sold','sum'),revenue=('gross_revenue_thb','sum'),profit_before_waste=('profit_before_waste','sum')
)
promo_history['profit_per_cup'] = promo_history['profit_before_waste']/promo_history['units']
promo_check = prices.merge(platforms,how='cross').merge(rate_codes,how='cross')
promo_check['selling_price'] = (promo_check['base_price_thb']*(1-promo_check['discount_pct'])).round(2)
promo_check['profit_before_waste'] = promo_check['selling_price']*(1-promo_check['commission_rate'])-promo_check['variable_cost']
promo_check['profit_after_waste'] = promo_check['profit_before_waste']-promo_check['waste_per_cup']
promo_check = promo_check.loc[promo_check['base_code'].ne('RC000')].copy()
promo_check['action'] = np.select(
    [promo_check['profit_before_waste'].lt(0),promo_check['profit_after_waste'].lt(0)],
    ['พัก/ลดส่วนลด: ต่ำกว่าต้นทุนผันแปร','ปรับส่วนลดและลดของเสีย'],
    default='ทดลองต่อและติดตามกำไรรวม'
)
display(promo_check[['sku','platform_name','rate_code_name','selling_price','profit_after_waste','action']])
# บันทึกผลโปรจริงและการประเมินโปร
actual = promo_history.rename(columns={
    'active_days':'actual_days','units':'actual_units','revenue':'actual_revenue',
    'profit_before_waste':'actual_profit_before_waste','profit_per_cup':'actual_profit_per_cup'
})
model = promo_check[['sku','platform_name','rate_code_name','commission_rate','selling_price','profit_before_waste','profit_after_waste','action']].rename(columns={
    'selling_price':'model_selling_price','profit_before_waste':'model_profit_before_waste',
    'profit_after_waste':'model_profit_after_waste'
})
promotions_csv = actual.merge(model,on=['sku','platform_name','rate_code_name'],how='left',validate='one_to_one')
promotions_csv.to_csv(folder/'promotions.csv', index=False, encoding='utf-8-sig')


,sku,platform_name,rate_code_name,selling_price,profit_after_waste,action
1,Guava,LINE MAN,Bundle Deal,41.65,6.82,ทดลองต่อและติดตามกำไรรวม
2,Guava,LINE MAN,Double Day New User,39.20,5.11,ทดลองต่อและติดตามกำไรรวม
3,Guava,LINE MAN,Seasonal Push,43.12,7.85,ทดลองต่อและติดตามกำไรรวม
5,Guava,Grab,Bundle Deal,41.65,9.32,ทดลองต่อและติดตามกำไรรวม
6,Guava,Grab,Double Day New User,39.20,7.46,ทดลองต่อและติดตามกำไรรวม
7,Guava,Grab,Seasonal Push,43.12,10.44,ทดลองต่อและติดตามกำไรรวม
9,MixedBerryPremium,LINE MAN,Bundle Deal,75.65,-27.22,พัก/ลดส่วนลด: ต่ำกว่าต้นทุนผันแปร
10,MixedBerryPremium,LINE MAN,Double Day New User,71.20,-30.33,พัก/ลดส่วนลด: ต่ำกว่าต้นทุนผันแปร
11,MixedBerryPremium,LINE MAN,Seasonal Push,78.32,-25.35,พัก/ลดส่วนลด: ต่ำกว่าต้นทุนผันแปร
13,MixedBerryPremium,Grab,Bundle Deal,75.65,-22.68,พัก/ลดส่วนลด: ต่ำกว่าต้นทุนผันแปร


---

## 4. Portfolio & budget performance
กำไรขั้นต้น = รายได้ − ผลไม้ − แพ็กเกจ − แรงงานผันแปร; กำไรจากการดำเนินงานหักคอมมิชชัน ของเสีย และ overhead เพิ่ม
งบรายเดือนระดับบริษัทคือรายได้ 2.7 ล้านบาทและ GP 7 แสนบาท โดยไม่ได้แจกแจงนิยาม GP ครบ จึงระบุนิยามที่ใช้ พร้อมแสดงกำไรหลังคอมมิชชัน/ของเสียให้ตรวจสอบ
P&L รวมต้นทุนของเสียที่ประมาณเพิ่ม 6 รายการแล้ว ตัวเลขส่วนนี้ควรแทนด้วยบันทึกจริงเมื่อได้รับข้อมูล Mixed Berry มีประวัติประมาณ 6 เดือนจึงดูต่อแก้วร่วมกับยอดรวม

In [7]:
amounts = ['units_sold','gross_revenue_thb','fruit_cost','pack_cost','labor_cost','commission','profit_before_waste']
pnl_kitchen = orders.groupby(['month','kitchen'],as_index=False)[amounts].sum()
pnl_kitchen = pnl_kitchen.merge(waste.groupby(['month','kitchen'],as_index=False).agg(waste_cost=('est_waste_cost_thb','sum')),on=['month','kitchen'],validate='one_to_one')
pnl_kitchen = pnl_kitchen.merge(overhead,on='kitchen',validate='many_to_one')
pnl_kitchen['gross_profit'] = pnl_kitchen['gross_revenue_thb']-pnl_kitchen[['fruit_cost','pack_cost','labor_cost']].sum(axis=1)
pnl_kitchen['contribution'] = pnl_kitchen['profit_before_waste']-pnl_kitchen['waste_cost']
pnl_kitchen['operating_profit'] = pnl_kitchen['contribution']-pnl_kitchen['overhead_thb']
totals = amounts+['waste_cost','overhead_thb','gross_profit','contribution','operating_profit']
pnl = pnl_kitchen.groupby('month',as_index=False)[totals].sum()
pnl = pnl.merge(budget,on='month',validate='one_to_one')
pnl['revenue_gap'] = pnl['gross_revenue_thb']-pnl['budget_revenue_thb']
pnl['gp_gap'] = pnl['gross_profit']-pnl['budget_gross_profit_thb']
pnl['contribution_gap'] = pnl['contribution']-pnl['budget_gross_profit_thb']
kitchens = pnl_kitchen.groupby('kitchen',as_index=False)[['gross_revenue_thb','contribution','overhead_thb','operating_profit']].sum()
menu = orders.groupby('sku',as_index=False).agg(units=('units_sold','sum'),revenue=('gross_revenue_thb','sum'),profit_before_waste=('profit_before_waste','sum'))
menu = menu.merge(waste.groupby('sku',as_index=False).agg(waste_cost=('est_waste_cost_thb','sum')),on='sku',validate='one_to_one')
menu['contribution'] = menu['profit_before_waste']-menu['waste_cost']
menu['profit_per_cup'] = menu['contribution']/menu['units']
display(pnl[['month','gross_revenue_thb','revenue_gap','gross_profit','gp_gap','contribution','contribution_gap','operating_profit']])
display(kitchens.sort_values('operating_profit'))
display(menu.sort_values('contribution'))

# ตารางเมนูรายเดือน ใช้เป็นข้อมูล CSV ให้แอป
menu_monthly = orders.groupby(['month','sku'],as_index=False).agg(
    units=('units_sold','sum'),revenue=('gross_revenue_thb','sum'),fruit=('fruit_cost','sum'),
    pack=('pack_cost','sum'),labor=('labor_cost','sum'),commission=('commission','sum'),
    profit_before_waste=('profit_before_waste','sum')
)
menu_monthly = menu_monthly.merge(waste.groupby(['month','sku'],as_index=False).agg(waste_cost=('est_waste_cost_thb','sum')),on=['month','sku'],validate='one_to_one')
menu_monthly = menu_monthly.merge(monthly_demand[['month','sku','days']],on=['month','sku'],validate='one_to_one')
menu_monthly = menu_monthly.merge(unit_costs[['sku','base_price_thb']],on='sku',validate='many_to_one')

# บันทึก P&L ผลแต่ละครัว และข้อมูลเมนูรายเดือน
pnl.to_csv(folder/'pnl.csv', index=False, encoding='utf-8-sig')
kitchens.to_csv(folder/'kitchens.csv', index=False, encoding='utf-8-sig')
menu_monthly.to_csv(folder/'menus.csv', index=False, encoding='utf-8-sig')

# ข้อมูลต้นฉบับระบุ exact duplicates แต่แสดงผลกรณีเก็บแถวเหล่านี้ด้วย
raw_keys = ['date','hour','day_of_week','kitchen','item_name_raw','price_thb','units_sold','gross_revenue_thb','rate_code']
duplicate_rows = pd.read_csv(cleaned/'review/duplicates.csv', parse_dates=['date'], encoding='utf-8-sig')
duplicate_rows = duplicate_rows.merge(orders[raw_keys+['profit_before_waste']], on=raw_keys, how='left', validate='many_to_one')
duplicate_main = duplicate_rows.dropna(subset=['profit_before_waste'])
duplicate_sensitivity = pd.DataFrame({
    'case':['ตัดแถวเหมือนกันตามคำอธิบายชุดข้อมูล','เก็บแถวเหมือนกันที่เป็นเมนูหลักทั้งหมด'],
    'rows':[len(orders),len(orders)+len(duplicate_main)],
    'revenue_thb':[pnl['gross_revenue_thb'].sum(),pnl['gross_revenue_thb'].sum()+duplicate_main['gross_revenue_thb'].sum()],
    'operating_profit_thb':[pnl['operating_profit'].sum(),pnl['operating_profit'].sum()+duplicate_main['profit_before_waste'].sum()]
})
display(duplicate_sensitivity)
duplicate_sensitivity.to_csv(folder/'duplicate_sensitivity.csv', index=False, encoding='utf-8-sig')


,month,gross_revenue_thb,revenue_gap,gross_profit,gp_gap,contribution,contribution_gap,operating_profit
0,2025-09-01,"1,913,361.14","-786,638.86","1,058,423.42","358,423.42","504,582.40","-195,417.60","-165,417.60"
1,2025-10-01,"1,967,214.49","-732,785.51","1,098,192.80","398,192.80","524,686.52","-175,313.48","-145,313.48"
2,2025-11-01,"2,254,716.05","-445,283.95","1,271,456.86","571,456.86","616,153.48","-83,846.52","-53,846.52"
3,2025-12-01,"2,425,157.05","-274,842.95","1,361,917.48","661,917.48","653,728.01","-46,271.99","-16,271.99"
4,2026-01-01,"2,458,094.35","-241,905.65","1,391,061.91","691,061.91","672,915.48","-27,084.52","2,915.48"
5,2026-02-01,"2,325,427.40","-374,572.60","1,305,864.45","605,864.45","626,312.61","-73,687.39","-43,687.39"
6,2026-03-01,"3,583,286.85","883,286.85","1,946,450.85","1,246,450.85","831,137.94","131,137.94","161,137.94"
7,2026-04-01,"4,029,174.75","1,329,174.75","2,250,193.82","1,550,193.82","1,007,879.78","307,879.78","337,879.78"
8,2026-05-01,"3,479,104.10","779,104.10","1,934,748.68","1,234,748.68","873,400.60","173,400.60","203,400.60"
9,2026-06-01,"2,302,538.53","-397,461.47","1,180,400.43","480,400.43","465,646.36","-234,353.64","-204,353.64"


,kitchen,gross_revenue_thb,contribution,overhead_thb,operating_profit
0,BKK_Ladprao,"4,603,743.10","1,059,032.16",1680000,"-620,967.84"
1,BKK_Sukhumvit,"7,702,657.17","1,913,447.48",2400000,"-486,552.52"
2,Pattaya_Beach,"8,639,507.43","2,132,445.09",1800000,"332,445.09"
3,Pattaya_Central,"10,201,024.87","2,531,425.58",2160000,"371,425.58"


,sku,units,revenue,profit_before_waste,waste_cost,contribution,profit_per_cup
1,MixedBerryPremium,28044,"2,439,126.88","281,076.79","334,936.82","-53,860.03",-1.92
2,PassionFruit,81416,"5,196,176.70","1,135,124.17","207,548.42","927,575.75",11.39
0,Guava,101504,"4,883,038.16","1,432,729.59","89,603.48","1,343,126.11",13.23
3,Pineapple,162168,"7,815,895.43","2,373,414.68","114,442.80","2,258,971.88",13.93
4,Watermelon,244991,"10,812,695.40","3,274,274.42","113,737.82","3,160,536.60",12.90


,case,rows,revenue_thb,operating_profit_thb
0,ตัดแถวเหมือนกันตามคำอธิบายชุดข้อมูล,123655,"31,146,932.57","-403,649.69"
1,เก็บแถวเหมือนกันที่เป็นเมนูหลักทั้งหมด,124023,"31,237,860.60","-377,917.42"


---

## 5. Inventory & 3-month outlook
ใช้ยอดเฉลี่ยรายวัน 28 วันล่าสุด แยกครัว/เมนู เพราะยอดขายช่วงล่าสุดต่ำกว่าช่วง มี.ค.–พ.ค.
ใช้ Rolling Forecast: ทุก 7 วันเติมข้อมูลจริงและคำนวณค่าเฉลี่ย 28 วันใหม่ พร้อมอัปเดตแผนวัตถุดิบและ P&L ล่วงหน้า 3 เดือน
ทดสอบการทยอยอัปเดตย้อนหลังหลายรอบ โดยแต่ละรอบใช้เฉพาะข้อมูลที่รู้ในขณะนั้น ผลนี้เป็นความแม่นของการทำนายช่วงใกล้หลังอัปเดต ไม่ใช่ความแม่นของการทำนาย 3 เดือนจากวันเดียว กรณีจำลองไม่ใช่ช่วงความเชื่อมั่น
เมื่อมีข้อมูลจริงใหม่ ให้รันหัวข้อ 1 เพื่อสร้าง cleaned_data ใหม่ แล้วรันหัวข้อ 2–6 หากไม่มีข้อมูลใหม่ ผลพยากรณ์จะไม่เปลี่ยนเอง

| กรณี | สมมติฐาน |
|---|---|
| ฐาน | ราคา ต้นทุน อัตราคอมมิชชันเฉลี่ย และของเสียตาม ส.ค. |
| แผนทดลอง | ราคา Watermelon/Pineapple/Guava/PassionFruit/Mixed Berry = 50/55/55/75/119 บาท; คงส่วนลดเดิมของ 4 เมนูแรก งดโปร Berry; ยอด 4 เมนูแรกลด 5%, Berry ลด 20%; อัตราของเสียลดครึ่งหนึ่ง; เป้าประหยัดครัวกรุงเทพฯ ครัวละ 15,000 บาท/เดือน |

ราคาและผลต่อยอดขายยังต้องทดลองจริง การลด overhead ต้องระบุรายการที่ประหยัดได้จริง
แผนสั่งใช้แก้วเทียบเท่าวัตถุดิบ ทบทวนทุกวัน ระยะส่ง 1 วัน buffer 0.5 วัน และอายุ 3 วันเป็นสมมติฐาน ใช้ FEFO และแบ่งเตรียมใกล้ช่วงขาย
แยกผลทดลองราคา ผลลดของเสีย และผลประหยัด overhead ออกจากกัน พร้อม stress test 18 กรณี ยอด Berry ลด 20/40/50% และอัตราของเสียลด 0/25/50% เป็นสมมติฐาน ไม่ใช่ผลที่วัดแล้ว
ตารางสั่งซื้ออ้างกรณีฐานที่ใช้ยอดล่าสุดและอัตราของเสียจริง เป้าสต็อก 2.5 วันเป็นตัวอย่างที่ต้องทดสอบด้วยสต็อกตามล็อตและการขาดสินค้า แผนทดลอง 14 วันและเงื่อนไขหยุดอยู่ใน REPORT.md

In [8]:
# ทดสอบวิธีเฉลี่ย 28 วัน โดยไม่ใช้ข้อมูลหลังวันตัด
errors = []
for start, end in [(pd.Timestamp('2026-08-01'),pd.Timestamp('2026-08-31')),(pd.Timestamp('2026-06-01'),pd.Timestamp('2026-08-31'))]:
    train = daily.loc[daily['date'].ge(start-pd.Timedelta(days=28)) & daily['date'].lt(start)]
    avg = train.groupby(['kitchen','sku'],as_index=False)['units_sold'].mean().rename(columns={'units_sold':'predicted'})
    test = daily.loc[daily['date'].between(start,end)].merge(avg,on=['kitchen','sku'],validate='many_to_one')
    errors.append(100*(test['units_sold']-test['predicted']).abs().sum()/test['units_sold'].sum())
print(f'WAPE ทำนาย ส.ค.: {errors[0]:.2f}% | ทำนาย มิ.ย.–ส.ค. รวดเดียว: {errors[1]:.2f}%')

# ทดสอบการอัปเดตทุก 7 วัน เทียบกับตรึงแผน 3 เดือนบนช่วงเดียวกัน
history_end = daily['date'].max()
latest_launch = daily.groupby('sku')['date'].min().max()
weekly_tests = []
for start in pd.date_range(latest_launch+pd.Timedelta(days=28), history_end, freq='MS'):
    stop = start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
    if stop > history_end:
        continue
    initial = daily.loc[daily['date'].between(start-pd.Timedelta(days=28), start-pd.Timedelta(days=1))]
    fixed = initial.groupby(['kitchen','sku'], as_index=False)['units_sold'].mean().rename(columns={'units_sold':'fixed_prediction'})
    for week in pd.date_range(start, stop, freq='7D'):
        train = daily.loc[daily['date'].between(week-pd.Timedelta(days=28), week-pd.Timedelta(days=1))]
        avg = train.groupby(['kitchen','sku'], as_index=False)['units_sold'].mean().rename(columns={'units_sold':'weekly_prediction'})
        actual = daily.loc[daily['date'].between(week, min(week+pd.Timedelta(days=6), stop))]
        test = actual.merge(fixed, on=['kitchen','sku'], validate='many_to_one').merge(avg, on=['kitchen','sku'], validate='many_to_one')
        test['origin'], test['issued_at'], test['train_start'] = start-pd.Timedelta(days=1), week-pd.Timedelta(days=1), week-pd.Timedelta(days=28)
        weekly_tests.append(test)
weekly_predictions = pd.concat(weekly_tests, ignore_index=True)
weekly_predictions['fixed_error'] = (weekly_predictions['units_sold']-weekly_predictions['fixed_prediction']).abs()
weekly_predictions['weekly_error'] = (weekly_predictions['units_sold']-weekly_predictions['weekly_prediction']).abs()
weekly_backtest = weekly_predictions.groupby('origin', as_index=False).agg(actual_cups=('units_sold','sum'), fixed_error=('fixed_error','sum'), weekly_error=('weekly_error','sum'))
weekly_backtest['fixed_wape_pct'] = 100*weekly_backtest['fixed_error']/weekly_backtest['actual_cups']
weekly_backtest['weekly_wape_pct'] = 100*weekly_backtest['weekly_error']/weekly_backtest['actual_cups']
fixed_wape = 100*weekly_backtest['fixed_error'].sum()/weekly_backtest['actual_cups'].sum()
weekly_wape = 100*weekly_backtest['weekly_error'].sum()/weekly_backtest['actual_cups'].sum()
display(weekly_backtest[['origin','fixed_wape_pct','weekly_wape_pct']])
print(f'WAPE รวม {len(weekly_backtest)} รอบ: ตรึงแผน 3 เดือน {fixed_wape:.2f}% | ทยอยอัปเดตทุกสัปดาห์ {weekly_wape:.2f}%')

# ฐานรายวันและต้นทุนเดือนล่าสุด แยกครัว/เมนู

forecast_as_of = daily['date'].max().normalize()
next_review = forecast_as_of + pd.Timedelta(days=7)
end = forecast_as_of
rates = daily.loc[daily['date'].ge(end-pd.Timedelta(days=27))].groupby(['kitchen','sku'],as_index=False)['units_sold'].mean().rename(columns={'units_sold':'cups_per_day'})
drivers = latest.groupby(['kitchen','sku'],as_index=False).agg(units=('units_sold','sum'),revenue=('gross_revenue_thb','sum'),fruit=('fruit_cost','sum'),pack=('pack_cost','sum'),labor=('labor_cost','sum'),commission=('commission','sum'))
drivers = drivers.merge(latest_waste.groupby(['kitchen','sku'],as_index=False).agg(wasted=('units_wasted','sum'),waste_cost=('est_waste_cost_thb','sum')),on=['kitchen','sku'],validate='one_to_one')
for column in ['revenue','fruit','pack','labor']:
    drivers[column+'_per_cup'] = drivers[column]/drivers['units']
drivers['commission_rate'] = drivers['commission']/drivers['revenue']
drivers['waste_rate'] = drivers['wasted']/(drivers['units']+drivers['wasted'])
drivers['waste_unit_cost'] = drivers['waste_cost']/drivers['wasted']
drivers = drivers.merge(unit_costs[['sku','base_price_thb']],on='sku',validate='many_to_one')
forecast_start = forecast_as_of + pd.Timedelta(days=1)
forecast_end = forecast_start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
months = pd.DataFrame({'date':pd.date_range(forecast_start, forecast_end)})
months['month'] = months['date'].dt.to_period('M').dt.to_timestamp()
months = months.groupby('month', as_index=False).agg(days=('date','size'))
future = rates.merge(drivers,on=['kitchen','sku'],validate='one_to_one').merge(months,how='cross')
trial_prices = {'Watermelon':50,'Pineapple':55,'Guava':55,'PassionFruit':75,'MixedBerryPremium':119}

plans = []
for scenario in ['ฐาน','แผนทดลอง']:
    f = future.copy()
    f['forecast_units'] = f['cups_per_day']*f['days']
    f['price'] = f['revenue_per_cup']
    f['loss_rate'] = f['waste_rate']
    if scenario == 'แผนทดลอง':
        f['forecast_units'] *= np.where(f['sku'].eq('MixedBerryPremium'),.8,.95)
        f['price'] *= f['sku'].map(trial_prices)/f['base_price_thb']
        f.loc[f['sku'].eq('MixedBerryPremium'),'price'] = 119
        f['loss_rate'] *= .5
    f['prepared_cups'] = f['forecast_units']/(1-f['loss_rate'])
    f['waste_cups'] = f['prepared_cups']-f['forecast_units']
    f['revenue_thb'] = f['forecast_units']*f['price']
    f['variable_cost_thb'] = f['forecast_units']*(f['fruit_per_cup']+f['pack_per_cup']+f['labor_per_cup'])
    f['commission_thb'] = f['revenue_thb']*f['commission_rate']
    f['waste_cost_thb'] = f['waste_cups']*f['waste_unit_cost']
    f['fruit_value_thb'] = f['forecast_units']*f['fruit_per_cup']+f['waste_cups']*(f['waste_unit_cost']-f['pack_per_cup'])
    f['contribution'] = f['revenue_thb']-f['variable_cost_thb']-f['commission_thb']-f['waste_cost_thb']
    f['scenario'] = scenario
    plans.append(f)
forecast_detail = pd.concat(plans,ignore_index=True)
forecast_kitchen = forecast_detail.groupby(['scenario','month','kitchen'],as_index=False)[['revenue_thb','variable_cost_thb','commission_thb','waste_cost_thb','contribution']].sum().merge(months,on='month',validate='many_to_one').merge(overhead,on='kitchen',validate='many_to_one')
# เมื่ออัปเดตกลางเดือน แบ่งค่าใช้จ่ายประจำตามวันในช่วงพยากรณ์
forecast_kitchen['period_fraction'] = forecast_kitchen['days']/forecast_kitchen['month'].dt.days_in_month
forecast_kitchen['overhead_thb'] *= forecast_kitchen['period_fraction']
forecast_kitchen['saving'] = np.where(forecast_kitchen['scenario'].eq('แผนทดลอง') & forecast_kitchen['kitchen'].str.startswith('BKK'),15000,0) * forecast_kitchen['period_fraction']
forecast_kitchen['profit_before_savings'] = forecast_kitchen['contribution']-forecast_kitchen['overhead_thb']
forecast_kitchen['operating_profit'] = forecast_kitchen['profit_before_savings']+forecast_kitchen['saving']
forecast = forecast_kitchen.groupby(['scenario','month'],as_index=False)[['revenue_thb','waste_cost_thb','profit_before_savings','saving','operating_profit']].sum()
inventory = forecast_detail.loc[forecast_detail['scenario'].eq('ฐาน')].groupby(['month','sku'],as_index=False)[['forecast_units','prepared_cups','waste_cups','fruit_value_thb']].sum()
order_plan = forecast_detail.loc[forecast_detail['scenario'].eq('ฐาน') & forecast_detail['month'].eq(months['month'].min()),['kitchen','sku','prepared_cups','days']].copy()
order_plan['daily_need'] = order_plan['prepared_cups']/order_plan['days']
order_plan['target_cups'] = np.ceil(order_plan['daily_need']*(1+1+.5))
order_plan['on_hand'], order_plan['on_order'] = 0, 0
order_plan['order_cups'] = (order_plan['target_cups']-order_plan['on_hand']-order_plan['on_order']).clip(lower=0)
order_plan = order_plan[['kitchen','sku','daily_need','target_cups','on_hand','on_order','order_cups']]
# ทดสอบสมมติฐานราคา/ของเสีย/overhead แยกกัน โดยไม่ถือว่าเงินประหยัดเกิดขึ้นแล้ว
sensitivity_rows = []
overhead_total = forecast_kitchen.loc[forecast_kitchen['scenario'].eq('ฐาน'),'overhead_thb'].sum()
period_months = (months['days']/months['month'].dt.days_in_month).sum()
for berry_drop in [20,40,50]:
    for waste_cut in [0,25,50]:
        stress = future.copy()
        stress['q'] = stress['cups_per_day']*stress['days']*np.where(stress['sku'].eq('MixedBerryPremium'),1-berry_drop/100,.95)
        stress['p'] = stress['revenue_per_cup']*stress['sku'].map(trial_prices)/stress['base_price_thb']
        stress.loc[stress['sku'].eq('MixedBerryPremium'),'p'] = 119
        stress['waste'] = stress['q']/(1-stress['waste_rate']*(1-waste_cut/100))-stress['q']
        profit_before_saving = (stress['q']*stress['p']*(1-stress['commission_rate'])
            -stress['q']*(stress['fruit_per_cup']+stress['pack_per_cup']+stress['labor_per_cup'])
            -stress['waste']*stress['waste_unit_cost']).sum()-overhead_total
        for monthly_saving in [0,30000]:
            sensitivity_rows.append({'berry_demand_drop_pct':berry_drop,'waste_reduction_pct':waste_cut,
                'monthly_overhead_saving':monthly_saving,'operating_profit_3m':profit_before_saving+monthly_saving*period_months})
scenario_sensitivity = pd.DataFrame(sensitivity_rows)
scenario_sensitivity.to_csv(folder/'scenario_sensitivity.csv', index=False, encoding='utf-8-sig')
central_case = scenario_sensitivity.loc[scenario_sensitivity['berry_demand_drop_pct'].eq(20)].set_index(['waste_reduction_pct','monthly_overhead_saving'])['operating_profit_3m']
plan_bridge = pd.DataFrame({'case':['ฐาน: ดำเนินต่อแบบเดิม','ทดลองราคา: ของเสียและ overhead เดิม',
    'เพิ่มการลดอัตราของเสีย 50%','เพิ่มการประหยัด overhead 30,000 บาท/เดือน'],
    'operating_profit_3m':[forecast.loc[forecast['scenario'].eq('ฐาน'),'operating_profit'].sum(),
        central_case.loc[(0,0)],central_case.loc[(50,0)],central_case.loc[(50,30000)]]})
display(plan_bridge)
display(scenario_sensitivity.loc[scenario_sensitivity['monthly_overhead_saving'].eq(0)].pivot(
    index='berry_demand_drop_pct',columns='waste_reduction_pct',values='operating_profit_3m'))

display(forecast)
display(inventory)
display(order_plan)
# บันทึกผลพยากรณ์สำหรับแอป
inventory_csv = forecast_detail.loc[forecast_detail['scenario'].eq('ฐาน'),[
    'scenario','month','kitchen','sku','days','forecast_units','prepared_cups','waste_cups','fruit_value_thb'
]]
forecast.to_csv(folder/'forecast.csv', index=False, encoding='utf-8-sig')
inventory_csv.to_csv(folder/'inventory.csv', index=False, encoding='utf-8-sig')

# รวมผลตรวจคุณภาพกับข้อมูลรอบพยากรณ์
notes = quality_notes + [{'section':'validation','item':'one_month','value':errors[0]},
              {'section':'validation','item':'three_months','value':errors[1]},
              {'section':'validation','item':'rolling_weekly','value':weekly_wape},
              {'section':'validation','item':'fixed_origins','value':fixed_wape},
              {'section':'as_of','item':'date','value':forecast_as_of.strftime('%Y-%m-%d')},
              {'section':'forecast_method','item':'window_days','value':28},
              {'section':'forecast_method','item':'update_days','value':7},
              {'section':'forecast_method','item':'next_review','value':next_review.strftime('%Y-%m-%d')},
              {'section':'forecast_method','item':'forecast_end','value':forecast_end.strftime('%Y-%m-%d')}]
notes.extend({'section':'weekly_backtest','item':row['origin'].strftime('%Y-%m-%d'),'value':'ทดสอบช่วง 3 เดือน',
              'fixed_wape_pct':row['fixed_wape_pct'],'weekly_wape_pct':row['weekly_wape_pct']} for _,row in weekly_backtest.iterrows())
pd.DataFrame(notes).to_csv(folder/'summary.csv', index=False, encoding='utf-8-sig')
print('บันทึก CSV ของแต่ละหัวข้อไว้ที่', folder.resolve())


WAPE ทำนาย ส.ค.: 14.29% | ทำนาย มิ.ย.–ส.ค. รวดเดียว: 52.22%
WAPE รวม 3 รอบ: ตรึงแผน 3 เดือน 42.37% | ทยอยอัปเดตทุกสัปดาห์ 20.07%
บันทึก CSV ของแต่ละหัวข้อไว้ที่ C:\Users\thewh\OneDrive - KMITL\แบบทดสอบ DABA (Intern)  Atmind Group\app_data


,origin,fixed_wape_pct,weekly_wape_pct
0,2026-03-31,24.43,20.98
1,2026-04-30,55.87,19.76
2,2026-05-31,52.22,19.14


,case,operating_profit_3m
0,ฐาน: ดำเนินต่อแบบเดิม,"-766,640.79"
1,ทดลองราคา: ของเสียและ overhead เดิม,"-167,017.81"
2,เพิ่มการลดอัตราของเสีย 50%,"-58,396.20"
3,"เพิ่มการประหยัด overhead 30,000 บาท/เดือน","31,603.80"


waste_reduction_pct,0,25,50
berry_demand_drop_pct,,,
20,"-167,017.81","-110,656.43","-58,396.20"
40,"-176,820.13","-126,940.49","-80,398.91"
50,"-181,721.29","-135,082.52","-91,400.26"


,scenario,month,revenue_thb,waste_cost_thb,profit_before_savings,saving,operating_profit
0,ฐาน,2026-09-01,"2,142,027.63","76,173.00","-260,101.36",0.00,"-260,101.36"
1,ฐาน,2026-10-01,"2,213,428.55","78,712.10","-246,438.07",0.00,"-246,438.07"
2,ฐาน,2026-11-01,"2,142,027.63","76,173.00","-260,101.36",0.00,"-260,101.36"
3,แผนทดลอง,2026-09-01,"2,293,696.09","31,151.25","-26,614.13","30,000.00","3,385.87"
4,แผนทดลอง,2026-10-01,"2,370,152.62","32,189.62","-5,167.94","30,000.00","24,832.06"
5,แผนทดลอง,2026-11-01,"2,293,696.09","31,151.25","-26,614.13","30,000.00","3,385.87"


,month,sku,forecast_units,prepared_cups,waste_cups,fruit_value_thb
0,2026-09-01,Guava,"6,917.14","7,314.20",397.06,"72,530.98"
1,2026-09-01,MixedBerryPremium,"2,469.64","3,085.06",615.42,"156,060.55"
2,2026-09-01,PassionFruit,"5,523.21","6,056.06",532.85,"168,795.66"
3,2026-09-01,Pineapple,"10,703.57","11,187.41",483.83,"112,915.67"
4,2026-09-01,Watermelon,"16,805.36","17,379.39",574.03,"137,797.96"
5,2026-10-01,Guava,"7,147.71","7,558.01",410.29,"74,948.68"
6,2026-10-01,MixedBerryPremium,"2,551.96","3,187.90",635.93,"161,262.56"
7,2026-10-01,PassionFruit,"5,707.32","6,257.93",550.61,"174,422.18"
8,2026-10-01,Pineapple,"11,060.36","11,560.32",499.96,"116,679.53"
9,2026-10-01,Watermelon,"17,365.54","17,958.70",593.17,"142,391.23"


,kitchen,sku,daily_need,target_cups,on_hand,on_order,order_cups
0,BKK_Ladprao,Guava,36.16,91.00,0,0,91.00
3,BKK_Ladprao,MixedBerryPremium,16.83,43.00,0,0,43.00
6,BKK_Ladprao,PassionFruit,30.14,76.00,0,0,76.00
9,BKK_Ladprao,Pineapple,57.93,145.00,0,0,145.00
12,BKK_Ladprao,Watermelon,85.79,215.00,0,0,215.00
15,BKK_Sukhumvit,Guava,59.47,149.00,0,0,149.00
18,BKK_Sukhumvit,MixedBerryPremium,24.92,63.00,0,0,63.00
21,BKK_Sukhumvit,PassionFruit,48.76,122.00,0,0,122.00
24,BKK_Sukhumvit,Pineapple,94.61,237.00,0,0,237.00
27,BKK_Sukhumvit,Watermelon,143.12,358.00,0,0,358.00
